# Bengaluru Climate Risk Scanner
## Notebook 05: Validation against the September 2022 flood (Phase 5)

Notebook 02 **predicted** where Bengaluru is flood-prone. Notebook 04 mapped where it **actually** flooded in September 2022. This notebook compares the two and turns the comparison into numbers you can quote.

| Measure | Question it answers | Good result |
|---|---|---|
| **Hit rate** | What share of the flooded area did the model call High or Medium? | Well above the share of land the model calls High or Medium (45%) |
| **High-only hit rate** | What share of the flooded area did the model call High? | Well above 15% (the share of land called High) |
| **Frequency ratio** | Per class: share of flooding ÷ share of land. 1 = no better than chance | Rises from Low to High; High clearly above 1 |
| **AUC** | Across all thresholds, how well does the flood risk index separate flooded from dry cells? | 0.5 = random, 0.7+ = useful, 0.8+ = strong |
| **Report hit rate** | How many news-reported flooded localities fall in High (or High/Medium) zones? | Well above the share of land in those classes |
| **False alarm share** | How much of the High zone did not flood in 2022? | Read with care (see below) |

**Why false alarms are expected.** One storm does not flood every susceptible place, and radar misses water between buildings. So a susceptibility map *should* contain High cells that stayed dry in 2022. Hit rate, frequency ratio and AUC are the fair measures; the false-alarm share is reported for honesty, not as a pass/fail.

**Outputs:** `validation_result.tif` (hit / miss / false alarm map for GeoServer), `validation_summary.csv`, charts in `maps/`, and a `validation` block in `config.json`.

## 1. Connect Google Drive and load the configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)
RAW, OUT, GPKG = CONFIG['paths']['raw'], CONFIG['paths']['processed'], CONFIG['paths']['gpkg']
MAPS = f'{PROJECT_DIR}/maps'
WORK = '/content/work'
for d in (MAPS, WORK):
    os.makedirs(d, exist_ok=True)
print('Flood model weights:', CONFIG['flood_model']['weights_used'])

## 2. Load the prediction and the observation
Everything is already on the same 30 m grid, so cells compare one to one. Only **city land** is evaluated: permanent lakes and cells outside the city are left out.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

def read(path, band=1):
    with rasterio.open(path) as src:
        a = src.read(band).astype('float32')
        if src.nodata is not None:
            a[a == src.nodata] = np.nan
        return a, src.transform, src.crs, src.profile

fri, transform, crs, profile = read(f'{OUT}/flood_risk.tif')
pred_class, _, _, _ = read(f'{OUT}/flood_class.tif')
obs, _, _, _ = read(f'{OUT}/flood_2022_observed.tif')
FACTORS = ['hand', 'slope', 'dist_lake', 'dist_drain', 'twi', 'builtup']
scores = {f: read(f'{OUT}/flood_factor_scores.tif', i + 1)[0] for i, f in enumerate(FACTORS)}

res = transform.a
cell_km2 = res * res / 1e6
land = np.isfinite(fri) & np.isin(pred_class, [1, 2, 3]) & np.isfinite(obs)
flooded = land & (obs == 1)
LEVELS = {1: 'Low', 2: 'Medium', 3: 'High'}

radar_ok = flooded.sum() > 0
print(f'City land evaluated: {land.sum() * cell_km2:,.1f} km²')
print(f'Observed flooded (radar): {flooded.sum() * cell_km2:,.2f} km² ({100 * flooded.sum() / land.sum():.2f}% of land)')
if not radar_ok:
    print('No radar-observed flood: the area-based checks (sections 3 to 6) are skipped; the report check (section 7) still runs.')

## 3. Hit rate and frequency ratio by class
For each predicted class: how much land it covers, how much observed flood falls in it, and the **frequency ratio** (FR) = share of flood ÷ share of land.

FR = 1 means the class floods exactly as often as a random place would. A good susceptibility map has FR below 1 for Low and clearly above 1 for High.

In [ ]:
rows = []
for c in (1, 2, 3):
    in_c = land & (pred_class == c)
    rows.append({'class': LEVELS[c],
                 'land_km2': in_c.sum() * cell_km2,
                 'flood_km2': (in_c & flooded).sum() * cell_km2})
by_class = pd.DataFrame(rows).set_index('class')
by_class['land_share_%'] = 100 * by_class['land_km2'] / by_class['land_km2'].sum()
if radar_ok:
    by_class['flood_share_%'] = 100 * by_class['flood_km2'] / by_class['flood_km2'].sum()
    by_class['frequency_ratio'] = by_class['flood_share_%'] / by_class['land_share_%']
    by_class['flooded_%_of_class'] = 100 * by_class['flood_km2'] / by_class['land_km2']

    hit_hm = by_class.loc[['Medium', 'High'], 'flood_share_%'].sum()
    hit_h = by_class.loc['High', 'flood_share_%']
    base_hm = by_class.loc[['Medium', 'High'], 'land_share_%'].sum()
    base_h = by_class.loc['High', 'land_share_%']
    false_alarm = 100 - by_class.loc['High', 'flooded_%_of_class']
    print(by_class.round(2).to_string())
    print(f'\nHit rate (High or Medium): {hit_hm:.1f}% of flooded area   (random map: {base_hm:.1f}%)')
    print(f'Hit rate (High only):      {hit_h:.1f}% of flooded area   (random map: {base_h:.1f}%)')
    print(f'False alarm share:         {false_alarm:.1f}% of the High zone did not flood in 2022')
else:
    hit_hm = hit_h = false_alarm = None
    print(by_class.round(2).to_string())

## 4. AUC: how well the index separates flooded from dry cells
The **ROC curve** tries every possible threshold on the flood risk index: at each one, it plots the share of flooded cells caught (true positive rate) against the share of dry cells wrongly flagged (false positive rate). The **area under the curve (AUC)** sums this up: 0.5 is a coin toss, 1.0 is perfect.

AUC is computed exactly from ranks (the Mann–Whitney formula), using every land cell.

In [ ]:
from scipy.stats import rankdata

def auc(score, label):
    """Probability that a random flooded cell has a higher score than a random dry cell."""
    r = rankdata(score)
    n1 = label.sum(); n0 = label.size - n1
    return float((r[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def roc_points(score, label, n=200):
    th = np.unique(np.quantile(score, np.linspace(0, 1, n)))[::-1]
    n1, n0 = label.sum(), (~label).sum()
    tpr = [((score >= t) & label).sum() / n1 for t in th]
    fpr = [((score >= t) & ~label).sum() / n0 for t in th]
    return np.r_[0, fpr, 1], np.r_[0, tpr, 1]

if radar_ok:
    y = flooded[land]
    fri_land = fri[land]
    auc_fri = auc(fri_land, y)
    print(f'AUC of the flood risk index: {auc_fri:.3f}')

    single = {f: auc(scores[f][land], y) for f in FACTORS}
    print('\nAUC of each factor on its own (the combined index should beat most of them):')
    for f, a in sorted(single.items(), key=lambda kv: -kv[1]):
        print(f'  {f:<11} {a:.3f}')
else:
    auc_fri, single = None, {}

## 5. Which factors matter? Weight sensitivity
Two quick experiments on the same 2022 data:

- **Leave one out**: drop one factor (share its weight among the others) and see how much AUC falls. A big fall means the factor carries real signal.
- **Alternative weight sets**: equal weights, and the current config weights.

**Careful with tuning.** Changing weights until they fit 2022 perfectly would overfit one storm. Only change weights if a difference is large (say, AUC improves by more than 0.02) **and** makes physical sense; write down why. This cell never changes the model by itself.

In [ ]:
W = CONFIG['flood_model']['weights_used']

def auc_for(weights):
    tot = sum(weights.values())
    s = sum((weights[f] / tot) * scores[f][land] for f in FACTORS if weights[f] > 0)
    return auc(s, y)

if radar_ok:
    sens = [{'experiment': 'Config weights (current model)', 'auc': auc_for(W)},
            {'experiment': 'Equal weights', 'auc': auc_for({f: 1 for f in FACTORS})}]
    for f in FACTORS:
        sens.append({'experiment': f'Without {f}', 'auc': auc_for({**W, f: 0})})
    sens = pd.DataFrame(sens)
    sens['change_vs_current'] = sens['auc'] - sens.loc[0, 'auc']
    print(sens.round(3).to_string(index=False))
else:
    sens = pd.DataFrame()

## 6. Validation map: hits, misses and false alarms
Each city land cell gets one label, saved as `validation_result.tif` for GeoServer:

| Value | Label | Meaning |
|---|---|---|
| 1 | Hit | Flooded in 2022 and predicted High or Medium |
| 2 | Miss | Flooded in 2022 but predicted Low |
| 3 | False alarm | Predicted High but not seen flooded in 2022 |
| 0 | Other land | Everything else |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes = gpd.read_file(GPKG, layer='lakes').to_crs(crs)

val = np.full(pred_class.shape, 255, dtype='uint8')
val[land] = 0
val[land & flooded & np.isin(pred_class, [2, 3])] = 1
val[land & flooded & (pred_class == 1)] = 2
val[land & ~flooded & (pred_class == 3)] = 3

prof = profile.copy()
prof.update(driver='GTiff', dtype='uint8', count=1, nodata=255, compress='deflate', tiled=True, blockxsize=256, blockysize=256)
with rasterio.open(f'{WORK}/validation_result.tif', 'w', **prof) as dst:
    dst.write(val, 1)
shutil.copy(f'{WORK}/validation_result.tif', f'{OUT}/validation_result.tif')
print('Saved', f'{OUT}/validation_result.tif')
for v, name in ((1, 'Hit'), (2, 'Miss'), (3, 'False alarm')):
    print(f'  {name:<12} {(val == v).sum() * cell_km2:8.2f} km²')

ny, nx = val.shape
extent = [transform.c, transform.c + nx * res, transform.f - ny * res, transform.f]
colors = ['#eef1f4', '#1b7837', '#d6604d', '#f4c26b']
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.where(val == 255, np.nan, val), cmap=ListedColormap(colors), norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4),
          extent=extent, interpolation='nearest')
lakes.plot(ax=ax, color='#b7c7d6', zorder=3)
boundary.boundary.plot(ax=ax, color='#1f2328', linewidth=0.8, zorder=4)
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(colors[1:], ['Hit: flooded, predicted High/Medium',
          'Miss: flooded, predicted Low', 'False alarm: predicted High, dry in 2022'])] +
          [Patch(color='#b7c7d6', label='Lake')], loc='lower left', fontsize=8, framealpha=0.95)
ax.set_title('Flood model vs the September 2022 flood (Sentinel-1)', loc='left', fontweight='bold')
ax.set_axis_off()
fig.savefig(f'{MAPS}/validation_map.png', dpi=250, bbox_inches='tight', facecolor='white')
plt.show()

## 7. News-reported flooded localities
Each reported point is checked against the predicted class at the point, and against the **highest class within about 100 m** (reported locations are approximate, and a flooded street usually sits next to the low ground the model flags). Results are given for all points and for **verified points only**.

The "random map" column is the share of land in that class: what you'd expect if points landed by chance.

In [ ]:
from scipy import ndimage

reports = gpd.read_file(GPKG, layer='flood_2022_reports').to_crs(crs)
reports['verified'] = reports['verified'].astype(str).str.upper().isin(['TRUE', '1', 'YES'])

cls = np.nan_to_num(np.where(land, pred_class, 0)).astype('uint8')
cls_near = ndimage.maximum_filter(cls, size=7)            # 7 x 7 cells = about 100 m around the point
rows_, cols_ = rasterio.transform.rowcol(transform, reports.geometry.x.values, reports.geometry.y.values)
rows_, cols_ = np.asarray(rows_), np.asarray(cols_)
ok = (rows_ >= 0) & (rows_ < cls.shape[0]) & (cols_ >= 0) & (cols_ < cls.shape[1])
reports['class_at_point'] = np.where(ok, cls[np.clip(rows_, 0, cls.shape[0] - 1), np.clip(cols_, 0, cls.shape[1] - 1)], 0)
reports['class_within_100m'] = np.where(ok, cls_near[np.clip(rows_, 0, cls.shape[0] - 1), np.clip(cols_, 0, cls.shape[1] - 1)], 0)
reports['level_within_100m'] = reports['class_within_100m'].map({0: 'outside / water', **LEVELS})
inside = reports[reports['class_within_100m'] > 0]

def rate(df, classes, col):
    return 100 * df[col].isin(classes).mean() if len(df) else float('nan')

share = lambda classes: 100 * np.isin(pred_class[land], classes).mean()
point_table = pd.DataFrame({
    'points': [len(inside), int(inside['verified'].sum())],
    'High (at point) %': [rate(inside, [3], 'class_at_point'), rate(inside[inside['verified']], [3], 'class_at_point')],
    'High within 100 m %': [rate(inside, [3], 'class_within_100m'), rate(inside[inside['verified']], [3], 'class_within_100m')],
    'High/Medium within 100 m %': [rate(inside, [2, 3], 'class_within_100m'), rate(inside[inside['verified']], [2, 3], 'class_within_100m')],
}, index=['All reported points', 'Verified points only']).round(1)
print(point_table.to_string())
print(f'\nRandom map: High {share([3]):.1f}% of land, High/Medium {share([2, 3]):.1f}% of land')
print('\nPoint by point:')
print(reports[['name', 'verified', 'level_within_100m']].to_string(index=False))
if reports['verified'].sum() == 0:
    print('\nNo verified points yet: verify flood_2022_reports.csv (Notebook 04, section 10) before quoting these numbers.')

## 8. Charts for the report
Left: the frequency ratio by class (the bar for High should stand clearly above the dashed line at 1). Right: the ROC curve with its AUC, against the diagonal a random map would follow.

In [ ]:
if radar_ok:
    INK, MUTED, GRID = '#1f2328', '#6e7781', '#d0d7de'
    BAR = ['#c4e6b8', '#5fbcc0', '#16306b']
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.8))

    fr = by_class['frequency_ratio']
    bars = a.bar(fr.index, fr.values, color=BAR, width=0.55, edgecolor='white', linewidth=2)
    a.axhline(1, color=MUTED, lw=1, ls='--')
    a.text(-0.42, 1.03, 'random = 1', color=MUTED, fontsize=8, ha='left', va='bottom')
    for rect, v in zip(bars, fr.values):
        a.text(rect.get_x() + rect.get_width() / 2, v + 0.04, f'{v:.2f}', ha='center', va='bottom', fontsize=9, color=INK)
    a.set_title('Frequency ratio by predicted class', loc='left', fontweight='bold', color=INK)
    a.set_ylabel('Share of 2022 flood ÷ share of land', color=MUTED)

    fpr, tpr = roc_points(fri_land, y)
    b.plot([0, 1], [0, 1], color=MUTED, lw=1, ls='--')
    b.plot(fpr, tpr, color='#16306b', lw=2)
    b.text(0.62, 0.18, f'AUC = {auc_fri:.2f}', fontsize=12, fontweight='bold', color='#16306b')
    b.text(0.62, 0.11, 'random = 0.50', fontsize=8, color=MUTED)
    b.set_xlim(0, 1); b.set_ylim(0, 1); b.set_aspect('equal')
    b.set_xlabel('False positive rate (dry cells flagged)', color=MUTED)
    b.set_ylabel('True positive rate (flooded cells caught)', color=MUTED)
    b.set_title('ROC curve of the flood risk index', loc='left', fontweight='bold', color=INK)

    for ax in (a, b):
        ax.grid(axis='y', color=GRID, lw=0.6); ax.set_axisbelow(True)
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        for s in ('left', 'bottom'):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED)
    plt.tight_layout()
    fig.savefig(f'{MAPS}/validation_charts.png', dpi=250, facecolor='white')
    plt.show()

## 9. Record the results
Saves a one-row-per-measure `validation_summary.csv` and a `validation` block in `config.json`. The printout at the end is the sentence for your README, report and interviews.

In [ ]:
from datetime import datetime, timezone

v_pts = inside[inside['verified']]
summary = {
    'observed_flood_km2': round(float(flooded.sum() * cell_km2), 3),
    'hit_rate_high_medium_pct': None if hit_hm is None else round(float(hit_hm), 1),
    'hit_rate_high_pct': None if hit_h is None else round(float(hit_h), 1),
    'land_share_high_medium_pct': round(float(share([2, 3])), 1),
    'land_share_high_pct': round(float(share([3])), 1),
    'false_alarm_share_pct': None if false_alarm is None else round(float(false_alarm), 1),
    'frequency_ratio': None if not radar_ok else {k: round(float(v), 2) for k, v in by_class['frequency_ratio'].items()},
    'auc_fri': None if auc_fri is None else round(auc_fri, 3),
    'auc_single_factors': {k: round(v, 3) for k, v in single.items()},
    'report_points': int(len(inside)), 'report_points_verified': int(len(v_pts)),
    'report_high_within_100m_pct_verified': None if not len(v_pts) else round(rate(v_pts, [3], 'class_within_100m'), 1),
    'report_high_medium_within_100m_pct_verified': None if not len(v_pts) else round(rate(v_pts, [2, 3], 'class_within_100m'), 1),
    'run_at': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
}
CONFIG['validation'] = summary
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)}).to_csv(f'{OUT}/validation_summary.csv', header=['value'])
print(json.dumps(summary, indent=2))

print('\n--- For your README and interviews ---')
if radar_ok:
    print(f'Against the September 2022 flood mapped from Sentinel-1, the model placed {summary["hit_rate_high_medium_pct"]}% '
          f'of the flooded area in its High or Medium zones, which cover {summary["land_share_high_medium_pct"]}% of the city '
          f'(AUC {summary["auc_fri"]}).')
if len(v_pts):
    print(f'{summary["report_high_medium_within_100m_pct_verified"]}% of {len(v_pts)} verified news-reported flood locations '
          f'fall within 100 m of a High or Medium zone.')
print('\nPhase 5 complete. Next: Phase 6, publishing the layers in GeoServer.')